# Part 1, Topic 4: Authenticated AES Bootloader

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *In the previous lab, we saw how glitching might be used to get a bootloader to dump its entire memory by glitching its response to a command.*

*In this lab, we'll again be targeting a bootloader. This time, however, we have a different objective. Instead of trying to read firmware, we'll be trying to bypass the target's authentication in order to upload our own firmware.*

**LEARNING OUTCOMES:**

* Using glitching to bypass authentication of a bootloader command
* Performing a CPA attack using only traces that bypassed authentication

**Prerequisites**
This lab requires you to do a CPA attack on the bootloader. If you haven't yet, it's recommended that you run through SCA101 before attempting this lab.

## Target Bootloader

Throughout SCA101 (and even SCA201), we've focused exclusively on unauthenticated encryption. In real systems, this is often a very bad idea. If you're interested, the following blog post outlines some basic attacks on unauthenticated encryption: https://cybergibbons.com/reverse-engineering-2/why-is-unauthenticated-encryption-insecure/.

For our bootloader example, it would be catastrophic if we were able to modify the resulting firmware. We could then, for example, inject malicious code to dump out the firmware or encryption key. To try to prevent this, our target appends a message authentication code to each encrypted frame in an [Encrypt-then-MAC scheme](https://en.wikipedia.org/wiki/Authenticated_encryption#Encrypt-then-MAC). To keep things simple, the "hash function" is just an 8-bit CRC. When calculating the MAC (Message Authentication Code), the ciphertext is appended to a 64-bit key. Note  that this scheme is **not** secure; however, it is simple and it lets us reuse code from other parts of ChipWhisperer. In total, the whole frame this looks like:

* 16 bytes of encrypted (AES128-CBC) firmware
* 1 byte MAC

The frame is then wrapped in our standard SimpleSerial packet we'll be using the 'p' command (or 0x01 on SSV2) to send 
a frame to the target.

To start, setup the hardware as usual:

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWHUSKY'
SS_VER = 'SS_VER_2_1'

In [2]:
%run "../../Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.gain                          changed from 0                         to 22                       
scope.gain.db                            changed from 15.0                      to 25.091743119266056       
scope.adc.samples                        changed from 131124                    to 5000                     
scope.clock.clkgen_freq                  changed from 0                         to 7363636.363636363        
scope.clock.adc_freq                     changed from 0                         to 29454545.454545453       
scope.clock.adc_rate                     changed from 0.0                       to 29454545.454545453       
scope.io.tio1                            changed from serial_tx                 to serial_rx                
scope.io.tio2                            changed from serial_rx                 to serial_tx                
scope.io.hs2                             changed from None                      to clkgen            

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-aes-bootloader
make PLATFORM=$1 CRYPTO_TARGET=TINYAES128C SS_VER=$2

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
.
Welcome to another exciting ChipWhisperer target build!!
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWHUSKY 
.
Compiling:
-en     bootloader.c ...
-e Done!
.
Compiling:
-en     .././simpleserial/simpleserial.c ...
-e Done!
.
Compiling:
-en     .././hal/hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/startup_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sam4s_hal.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/uart.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pio.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/system_sam4s.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/sysclk.c ...
-e Done!
.
Compiling:
-en     .././hal//sam4s/pmc.c ...
-e Done!
.
Compiling:
-en

In [4]:
fw_path = "../../../firmware/mcu/simpleserial-aes-bootloader/simpleserial-bootloader-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
target.reset_comms()

In [5]:
import os
print("cwd repr:", repr(os.getcwd()))
print("fw_path repr:", repr(fw_path))
print("PLATFORM repr:", repr(PLATFORM))
print("abspath 1:", repr(os.path.abspath(fw_path)))
print("abspath 2:", repr(os.path.abspath(fw_path)))  # run twice, check consistency
print("exists:", os.path.exists(fw_path))

cwd repr: 'C:\\Users\\lewis\\ChipWhisperer\\chipwhisperer\\jupyter\\courses\\fault101'
fw_path repr: '../../../firmware/mcu/simpleserial-aes-bootloader/simpleserial-bootloader-CWHUSKY.hex'
PLATFORM repr: 'CWHUSKY'
abspath 1: 'C:\\Users\\lewis\\ChipWhisperer\\chipwhisperer\\firmware\\mcu\\simpleserial-aes-bootloader\\simpleserial-bootloader-CWHUSKY.hex'
abspath 2: 'C:\\Users\\lewis\\ChipWhisperer\\chipwhisperer\\firmware\\mcu\\simpleserial-aes-bootloader\\simpleserial-bootloader-CWHUSKY.hex'
exists: True


We can pull in the CRC calculation from the SimpleSerial2 class:

In [6]:
calc_crc = cw.targets.SimpleSerial2._calc_crc

In [7]:
print(calc_crc([0x00, 0x12]))

def bootloader_calc_crc(buf):
    mac_key = [0x2b, 0x7e, 0x15, 0x16, 0x28, 0xae, 0xd2, 0xa6]
    crc_buf = bytearray(mac_key)
    crc_buf.extend(buf)
    return calc_crc(crc_buf)

51


In [8]:
ktp = cw.ktp.Basic()
key, text = ktp.next()

print(bootloader_calc_crc(text))

179


We'll probably crash the target a few times while we're trying some glitching. Create a function to reset the target:

In [9]:
def reboot_flush():
    reset_target(scope)
    target.reset_comms()
    target.simpleserial_write(0x00, bytearray(range(16)))
    target.simpleserial_wait_ack()

We want to catch both the MAC calculation and the beginning of AES, so we'll set `adc.samples` to its max:

In [10]:
scope.adc.samples = 24400
scope.adc.offset = 5000

## Communication

To start off, we need to inititalize the bootloader with an IV:

In [11]:
target.simpleserial_write(0x00, bytearray(range(16)))
target.simpleserial_wait_ack()

bytearray(b'\x00')

The bootloader protects against rewriting the IV, so if you rerun the block above the target won't update the IV and it'll return `0x12` as an error code.

Let's start by seeing what the power traces for a correctly authenticated message:

In [12]:
#Do glitch loop
key, text = ktp.next()
text.extend([bootloader_calc_crc(text)])
scope.arm()
target.simpleserial_write("p", text)
scope.capture()
val = target.simpleserial_wait_ack() # For loop check

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)
cw.plot(scope.get_last_trace())

bytearray(b'\x00')


:Curve   [x]   (y)

and an incorrectly authenticated one:

In [13]:
#Do glitch loop
key, text = ktp.next()
text.extend([0x00])
scope.arm()
target.simpleserial_write("p", text)
scope.capture()
val = target.simpleserial_wait_ack()#For loop check

#print(bytearray(val['full_response'].encode('latin-1')))
print(val)
cw.plot(scope.get_last_trace())

(ChipWhisperer Target ERROR|File SimpleSerial2.py:295) Device reported error Unknown error (0x11)
(ChipWhisperer Target ERROR|File SimpleSerial2.py:296) Full packet: CWbytearray(b'00 65 01 11 0f 00')


bytearray(b'\x11')


:Curve   [x]   (y)

As you can see, the target only decrypts our message if it's got the right MAC. If the target didn't tell us that our MAC was wrong, we could use to help this determine if we got our glitch or not.

Next, we need to setup our glitch. By this point, you should have some fairly reliable settings you can use:

In [16]:
scope.cglitch_setup(default_setup=False)  # make sure default_setup is off so we don't overwrite our adc settings
gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

IntText(value=0, description='success count:', disabled=True)

IntText(value=0, description='reset count:', disabled=True)

IntText(value=0, description='normal count:', disabled=True)

FloatSlider(value=0.0, continuous_update=False, description='width setting:', disabled=True, max=10.0, readout…

FloatSlider(value=0.0, continuous_update=False, description='offset setting:', disabled=True, max=10.0, readou…

FloatSlider(value=0.0, continuous_update=False, description='ext_offset setting:', disabled=True, max=10.0, re…

In [17]:
gc.glitch_plot(plotdots={"success":"+g", "reset":"xr", "normal":None}, bufferlen=int(100E6))

Parameter name clashes for keys ['data'] 
Parameter name clashes for keys ['data'] 
Parameter name clashes for keys ['data']

:DynamicMap   []
   :Overlay
      .Points.I  :Points   [width,offset]
      .Points.II :Points   [width,offset]

In [18]:
print("trig_count:", scope.adc.trig_count)

gc.set_range("width", 0, 600)
gc.set_range("offset", 2200, 2400)
gc.set_global_step([100, 50])
gc.set_range("ext_offset", 0, scope.adc.trig_count)
gc.set_step("ext_offset", 20)

trig_count: 7572


We might get some "successful" glitches here that don't actually bypass the authentication. Therefore, let's get say 10 or so potentially successful glitches then check them afterwards for a good offset.

In [37]:
scope.adc.timeout = 0.1
reboot_flush()
target.reset_comms()
x = []
req_encs = 10
encs = 0

project = cw.create_project("projects/test_bootloader", overwrite=True)

# --- Calibrate: what does a normal (non-glitched, correctly rejected) response look like? ---
baseline_vals = []
for _ in range(5):
    reboot_flush()
    target.reset_comms()
    key, text = ktp.next()
    text.extend([0x00])  # wrong MAC byte -> should be cleanly rejected, no glitch involved
    scope.arm()
    target.simpleserial_write("p", text)
    ret = scope.capture()
    val = target.simpleserial_read('e', 1, ack=False)
    if val is not None:
        baseline_vals.append(val[0])
print("Baseline (normal) response byte(s) seen:", baseline_vals)
if not baseline_vals:
    raise RuntimeError("Never got a valid baseline response - check target/comms before glitching")
normal_gcnt = max(set(baseline_vals), key=baseline_vals.count)  # most common value
print("Using normal_gcnt =", normal_gcnt)

reboot_flush()
target.reset_comms()

for glitch_setting in gc.glitch_values():

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    if encs > req_encs:
        break

    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.ext_offset = glitch_setting[2]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")
        reboot_flush()
        #target.reset_comms()

    #Do glitch loop
    key, text = ktp.next()
    cpy_text = bytearray(text)
    text.extend([0x00])
    scope.arm()
    target.simpleserial_write("p", text)
    ret = scope.capture()
    val = target.simpleserial_read('e', 1, ack=False)
    #print(val)

    if ret:  # here the trigger never went high - sometimes the target is stil crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        target.reset_comms()
    else:
        if val is None:  # invalid/garbled response
            gc.add("reset")
        else:
            # gcnt is the loop counter / status byte
            gcnt = val[0]

            if gcnt == normal_gcnt:  # normal response
                gc.add("normal")
            else:  # glitch!!!
                gc.add("success")
                print(f"Loc: {scope.glitch.ext_offset}, Glitch number: {encs}")
                x.append(scope.get_last_trace())
                trace = cw.Trace(scope.get_last_trace(), cpy_text, key, key)
                project.traces.append(trace)
                encs += 1

Baseline (normal) response byte(s) seen: [17, 17, 17, 17, 17]
Using normal_gcnt = 17
Loc: 0, Glitch number: 0


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 


Trigger still high!
Loc: 211, Glitch number: 1


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 


Trigger still high!
Loc: 267, Glitch number: 2


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 


... (16 outputs left out)

Trigger still high!
Loc: 1152, Glitch number: 6


(ChipWhisperer Target WARNING|File SimpleSerial2.py:533) Read timed out: 


Trigger still high!


KeyboardInterrupt: 

Let's plot these glitches:

In [38]:
plot = cw.plot(x[0])
for y in x[1:]:
    plot += cw.plot(y)
plot

:Layout
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)
   .Curve.V   :Curve   [x]   (y)
   .Curve.VI  :Curve   [x]   (y)
   .Curve.VII :Curve   [x]   (y)

Hopefully you got at least one glitch that looks like the authenticated version of the encryption. We were able to use the return message to narrow down our glitches, but these other glitches shows that power traces are still very useful here.

Now that we know where to insert a glitch to bypass the authentication, we could use one of those attacks on unauthenticated encryption. Instead, let's do a CPA attack. That way, we'll be able to insert as much code without the limitations of those other attacks. Start by finding an offset that bypasses the authentication:

In [39]:
cw.plot(x[0])

:Curve   [x]   (y)

Next, let's repeat the attack, with the objective this time being to gather enough encryptions to recover the key:

In [40]:
results = gc.calc(sort="success_rate")
results[:5]

[((0, 2300, 1152),
  {'total': 1,
   'success': 1,
   'success_rate': 1.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((0, 2300, 993),
  {'total': 1,
   'success': 1,
   'success_rate': 1.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((0, 2300, 956),
  {'total': 1,
   'success': 1,
   'success_rate': 1.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((0, 2300, 551),
  {'total': 1,
   'success': 1,
   'success_rate': 1.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 0,
   'normal_rate': 0.0}),
 ((0, 2300, 267),
  {'total': 1,
   'success': 1,
   'success_rate': 1.0,
   'reset': 0,
   'reset_rate': 0.0,
   'normal': 0,
   'normal_rate': 0.0})]

In [41]:
%%time
scope.glitch.ext_offset = 280

gc.set_range("ext_offset", 0, 10)
#gc.set_global_step([1])
scope.glitch.repeat = 1

scope.adc.timeout = 0.1

reboot_flush()
target.reset_comms()
x = []
req_encs = 100
scope.adc.offset = 10000
encs = 0

gc.set_range("width", 0, 0)        # fixed at your known-good value, not a scan
gc.set_range("offset", 2300, 2300) # same
gc.set_global_step(1)
gc.set_range("ext_offset", 0, 20000)   # was 0, req_encs - too few tries for a ~1% hit rate
gc.set_step("ext_offset", 1)

project = cw.create_project("projects/test_bootloader", overwrite=True)

for glitch_setting in gc.glitch_values():

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    if encs > req_encs:
        break

    # optional: you can speed up the loop by checking if the trigger never went low
    #           (the target never called trigger_low();) via scope.adc.state
    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    if scope.adc.state:
        # can detect crash here (fast) before timing out (slow)
        print("Trigger still high!")
        gc.add("reset")
        reboot_flush()
        target.reset_comms()

    #Do glitch loop
    key, text = ktp.next()
    cpy_text = bytearray(text)
    text.extend([0x00])
    scope.arm()
    target.simpleserial_write("p", text)
    ret = scope.capture()
    val = target.simpleserial_read('e', 1, ack=False, timeout=50)
    #print(val)

    if ret: #here the trigger never went high - sometimes the target is stil crashed from a previous glitch
        print('Timeout - no trigger')
        gc.add("reset")

        #Device is slow to boot?
        reboot_flush()
        target.reset_comms()
    else:
        if val is None:  # change this to detect an invalid response
            gc.add("reset")
        else:
            # gcnt is the loop counter
            gcnt = val[0]
        
            if gcnt == normal_gcnt:  # normal response
                gc.add("normal")
            else:  # glitch!!!
                gc.add("success")
                print(scope.glitch.ext_offset)
                print(encs)
                x.append(scope.get_last_trace())
                trace = cw.Trace(scope.get_last_trace(), cpy_text, key, key)
                print(scope.glitch.width, scope.glitch.offset)
                project.traces.append(trace)
                encs += 1
                reboot_flush()          # <-- add this
                target.reset_comms()    # <-- and this

280
0
0 2300
280
1
0 2300
280
2
0 2300
280
3
0 2300
280
4
0 2300
280
5
0 2300
280
6
0 2300
280
7
0 2300
280
8
0 2300
280
9
0 2300
280
10
0 2300


KeyboardInterrupt: 

Let's do a sanity check to make sure these traces all have encryptions (this will take a while to plot so you might only want to plot a subset of the traces)

In [42]:
project.save()

In [43]:
plot = cw.plot([])
for y in x:
    plot *= cw.plot(y)
plot

:Overlay
   .Curve.I    :Curve   [x]   (y)
   .Curve.II   :Curve   [x]   (y)
   .Curve.III  :Curve   [x]   (y)
   .Curve.IV   :Curve   [x]   (y)
   .Curve.V    :Curve   [x]   (y)
   .Curve.VI   :Curve   [x]   (y)
   .Curve.VII  :Curve   [x]   (y)
   .Curve.VIII :Curve   [x]   (y)
   .Curve.IX   :Curve   [x]   (y)
   .Curve.X    :Curve   [x]   (y)
   .Curve.XI   :Curve   [x]   (y)
   .Curve.XII  :Curve   [x]   (y)

AES decryption and encryption are actually very similar, except with all the forward function replaced with inverse functions. For example, instead of the SBox, we have the inverse SBox, instead of MixColumns we have inverse MixColumns, etc. As such our attack will use the inverse SBox instead of the regular SBox (though we'll use the alternate model so the correct key gets highlighted in the table):

In [44]:
import chipwhisperer as cw
project = cw.open_project("projects/test_bootloader")

Depending on your target, you may need to resync your traces:

In [50]:
import chipwhisperer.analyzer as cwa
resync_traces = cwa.preprocessing.ResyncSAD(project)
resync_traces.ref_trace = 0
resync_traces.target_window = (100, 4900)   # leaves 100-sample margin for the ±100 max_shift
resync_traces.max_shift = 100
resync_analyzer = resync_traces.preprocess()

plot = cw.plot([])
for y in range(10):
    plot *= cw.plot(resync_analyzer.waves[y])
plot

100%|██████████████████████████████████████████████████████████████████████████████████| 11/11 [00:00<00:00, 40.69it/s]


:Overlay
   .Curve.I    :Curve   [x]   (y)
   .Curve.II   :Curve   [x]   (y)
   .Curve.III  :Curve   [x]   (y)
   .Curve.IV   :Curve   [x]   (y)
   .Curve.V    :Curve   [x]   (y)
   .Curve.VI   :Curve   [x]   (y)
   .Curve.VII  :Curve   [x]   (y)
   .Curve.VIII :Curve   [x]   (y)
   .Curve.IX   :Curve   [x]   (y)
   .Curve.X    :Curve   [x]   (y)
   .Curve.XI   :Curve   [x]   (y)

In [51]:
import chipwhisperer.analyzer as cwa
leak_model = cwa.leakage_models.inverse_sbox_output_alt
attack = cwa.cpa(project, leak_model)
results = attack.run(cwa.get_jupyter_callback(attack))

This won't actually recover our usual encryption key. Instead we've got the last round key. We can use ChipWhisperer to convert it back to the original key:

In [52]:
bytearray(cwa.aes_funcs.key_schedule_rounds(results.key_guess(), 10, 0))

bytearray(b'\x15\xf1Qt.\xb2\x0b\x8a\x1d\xd1\xb6l\xe4l\xd3\x89')

And we can see it's our usual 0x2b7e... key.

## Injecting Malicious (Advanced)

Now that we can both bypass the MAC, as well as encrypt messages, try creating a piece of malicious code to dump firmware. Following commands will be useful:

1. `set_addr()` sets the address to write to
1. `go()` will run code pointed to by `set_addr()`

In [53]:
scope.dis()
target.dis()

(ChipWhisperer Scope ERROR|File naeusbchip.py:113) Scope already disconnected!
